#<font color="blue"><b>******* Multi-Task PLM Fine-Tuning *******</b></font>

### <font color="blue"><b>Import Required Packages</b></font>

In [ ]:
import random
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import DistilBertTokenizerFast, DistilBertModel
from datasets import load_dataset

### <font color="blue"><b>Configurations</b></font>

In [ ]:
# ----------------------------- Config -----------------------------
SEED = 42
MODEL_NAME = "distilbert-base-uncased"
MAX_LEN = 128
BATCH_SIZE = 16
EPOCHS = 2                 # bump up for real training
TRAIN_SUBSET = 2000        # examples per task used for training (demo speed)
EVAL_SUBSET = 200          # examples per task used for quick eval
LR = 2e-5
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CHECKPOINT_PATH = "multitask_distilbert_finetuned.pt"  # where the fine-tuned weights are saved/loaded

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

### <font color="blue"><b>Task registry</b></font>

In [ ]:
# ----------------------------- Task registry -----------------------------
# Each task defines: name, type ("cls" or "reg"), num_labels (1 for reg),
# and a loader function returning (train_texts, train_labels, test_texts, test_labels).

def load_sst2():
    ds = load_dataset("nyu-mll/glue", "sst2")
    train = ds["train"].shuffle(seed=SEED).select(range(TRAIN_SUBSET))
    test = ds["validation"].shuffle(seed=SEED).select(range(EVAL_SUBSET))
    return (train["sentence"], train["label"],
            test["sentence"], test["label"])

def load_agnews():
    ds = load_dataset("wangrongsheng/ag_news")
    train = ds["train"].shuffle(seed=SEED).select(range(TRAIN_SUBSET))
    test = ds["test"].shuffle(seed=SEED).select(range(EVAL_SUBSET))
    return (train["text"], train["label"],
            test["text"], test["label"])

def load_cola():
    ds = load_dataset("nyu-mll/glue", "cola")
    n_train = min(TRAIN_SUBSET, len(ds["train"]))
    n_eval = min(EVAL_SUBSET, len(ds["validation"]))
    train = ds["train"].shuffle(seed=SEED).select(range(n_train))
    test = ds["validation"].shuffle(seed=SEED).select(range(n_eval))
    return (train["sentence"], train["label"],
            test["sentence"], test["label"])

def load_stsb():
    ds = load_dataset("nyu-mll/glue", "stsb")
    n_train = min(TRAIN_SUBSET, len(ds["train"]))
    n_eval = min(EVAL_SUBSET, len(ds["validation"]))
    train = ds["train"].shuffle(seed=SEED).select(range(n_train))
    test = ds["validation"].shuffle(seed=SEED).select(range(n_eval))
    # STS-B has sentence1 / sentence2 pairs; join them for a single-text input
    train_texts = [f"{a} [SEP] {b}" for a, b in zip(train["sentence1"], train["sentence2"])]
    test_texts = [f"{a} [SEP] {b}" for a, b in zip(test["sentence1"], test["sentence2"])]
    return (train_texts, train["label"], test_texts, test["label"])


TASKS = [
    {"name": "sst2",   "type": "cls", "num_labels": 2, "loader": load_sst2,
     "label_names": ["negative", "positive"]},
    {"name": "agnews", "type": "cls", "num_labels": 4, "loader": load_agnews,
     "label_names": ["World", "Sports", "Business", "Sci/Tech"]},
    {"name": "cola",   "type": "cls", "num_labels": 2, "loader": load_cola,
     "label_names": ["unacceptable", "acceptable"]},
    {"name": "stsb",   "type": "reg", "num_labels": 1, "loader": load_stsb,
     "label_names": None},
]

### <font color="blue"><b>Multi-Task Dataset creation</b></font>

In [ ]:
# ----------------------------- Dataset -----------------------------

class TaskDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, task_type):
        self.texts = list(texts)
        self.labels = list(labels)
        self.tokenizer = tokenizer
        self.task_type = task_type

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            truncation=True,
            max_length=MAX_LEN,
            padding="max_length",
            return_tensors="pt",
        )
        item = {k: v.squeeze(0) for k, v in enc.items()}
        if self.task_type == "reg":
            item["labels"] = torch.tensor(float(self.labels[idx]), dtype=torch.float)
        else:
            item["labels"] = torch.tensor(int(self.labels[idx]), dtype=torch.long)
        return item

### <font color="blue"><b>Model Class</b></font>

In [ ]:
# ----------------------------- Model -----------------------------

class MultiTaskDistilBert(nn.Module):
    """Shared DistilBERT encoder + one linear head per task."""

    def __init__(self, model_name, tasks):
        super().__init__()
        self.encoder = DistilBertModel.from_pretrained(model_name)
        hidden = self.encoder.config.dim
        self.dropout = nn.Dropout(0.1)
        self.heads = nn.ModuleDict({
            t["name"]: nn.Linear(hidden, t["num_labels"]) for t in tasks
        })

    def forward(self, input_ids, attention_mask, task_name):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        # DistilBERT has no pooler; use [CLS] token (first token) hidden state
        cls_repr = out.last_hidden_state[:, 0]
        cls_repr = self.dropout(cls_repr)
        logits = self.heads[task_name](cls_repr)
        return logits

### <font color="blue"><b>Train and Predict Functions</b></font>

In [ ]:
# ----------------------------- Train / Eval -----------------------------

def make_loader(dataset, shuffle):
    return DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=shuffle)

def train_multitask(model, task_loaders, epochs):
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
    cls_loss_fn = nn.CrossEntropyLoss()
    reg_loss_fn = nn.MSELoss()

    model.train()
    for epoch in range(epochs):
        # Round-robin over tasks so each batch trains a different task
        iterators = {name: iter(dl) for name, dl in task_loaders.items()}
        active = list(iterators.keys())
        step = 0
        while active:
            for name in list(active):
                try:
                    batch = next(iterators[name])
                except StopIteration:
                    active.remove(name)
                    continue

                task_type = next(t["type"] for t in TASKS if t["name"] == name)
                input_ids = batch["input_ids"].to(DEVICE)
                attention_mask = batch["attention_mask"].to(DEVICE)
                labels = batch["labels"].to(DEVICE)

                logits = model(input_ids, attention_mask, name)
                if task_type == "reg":
                    loss = reg_loss_fn(logits.squeeze(-1), labels)
                else:
                    loss = cls_loss_fn(logits, labels)

                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

                step += 1
                if step % 50 == 0:
                    print(f"epoch {epoch} step {step} task={name} loss={loss.item():.4f}")

@torch.no_grad()
def predict(model, text, tokenizer, task):
    model.eval()
    enc = tokenizer(text, truncation=True, max_length=MAX_LEN,
                     padding="max_length", return_tensors="pt")
    input_ids = enc["input_ids"].to(DEVICE)
    attention_mask = enc["attention_mask"].to(DEVICE)
    logits = model(input_ids, attention_mask, task["name"])
    if task["type"] == "reg":
        return round(logits.item(), 3)
    else:
        pred_idx = logits.argmax(dim=-1).item()
        return task["label_names"][pred_idx]

In [ ]:
# ----------------------------- Run-----------------------------

tokenizer = DistilBertTokenizerFast.from_pretrained(MODEL_NAME)
model = MultiTaskDistilBert(MODEL_NAME, TASKS).to(DEVICE)

task_loaders = {}
task_test_data = {}  # keep raw text/labels for the demo section

print("Loading datasets...")
for task in TASKS:
    train_texts, train_labels, test_texts, test_labels = task["loader"]()
    train_ds = TaskDataset(train_texts, train_labels, tokenizer, task["type"])
    task_loaders[task["name"]] = make_loader(train_ds, shuffle=True)
    task_test_data[task["name"]] = (list(test_texts), list(test_labels))
    print(f"  {task['name']}: {len(train_texts)} train / {len(test_texts)} test examples")

import os

if os.path.exists(CHECKPOINT_PATH):
    print(f"\nFound existing fine-tuned checkpoint at '{CHECKPOINT_PATH}' — loading it (skipping training).")
    model.load_state_dict(torch.load(CHECKPOINT_PATH, map_location=DEVICE))
    model.to(DEVICE)
else:
    print("\nNo fine-tuned checkpoint found — training from the pretrained base model...")
    train_multitask(model, task_loaders, EPOCHS)
    torch.save(model.state_dict(), CHECKPOINT_PATH)
    print(f"Saved fine-tuned weights to '{CHECKPOINT_PATH}'.")

print("\n" + "=" * 70)
print("DEMO: 5 test samples per task")
print("=" * 70)

for task in TASKS:
    print(f"\n--- Task: {task['name']} ---")
    texts, labels = task_test_data[task["name"]]
    idxs = random.sample(range(len(texts)), min(5, len(texts)))
    for i in idxs:
        text = texts[i]
        true_label = labels[i]
        pred = predict(model, text, tokenizer, task)
        if task["type"] != "reg":
            true_label = task["label_names"][true_label]
        display_text = text if len(text) <= 90 else text[:87] + "..."
        print(f"  text: {display_text}")
        print(f"    true: {true_label}   pred: {pred}\n")

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

...

stsb/test-00000-of-00001.parquet:   0%|          | 0.00/114k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/5749 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1500 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1379 [00:00<?, ? examples/s]

  stsb: 2000 train / 200 test examples

No fine-tuned checkpoint found — training from the pretrained base model...
epoch 0 step 50 task=agnews loss=1.1303
epoch 0 step 100 task=stsb loss=2.2036
epoch 0 step 150 task=agnews loss=0.9410
epoch 0 step 200 task=stsb loss=1.2062
epoch 0 step 250 task=agnews loss=0.5470
epoch 0 step 300 task=stsb loss=1.6887
epoch 0 step 350 task=agnews loss=0.7482
epoch 0 step 400 task=stsb loss=0.6244
epoch 0 step 450 task=agnews loss=0.5911
epoch 0 step 500 task=stsb loss=0.5239
epoch 1 step 50 task=agnews loss=0.6319
epoch 1 step 100 task=stsb loss=0.6425
epoch 1 step 150 task=agnews loss=0.2579
epoch 1 step 200 task=stsb loss=0.7555
epoch 1 step 250 task=agnews loss=0.1108
epoch 1 step 300 task=stsb loss=0.5708
epoch 1 step 350 task=agnews loss=0.6231
epoch 1 step 400 task=stsb loss=0.6409
epoch 1 step 450 task=agnews loss=0.0919
epoch 1 step 500 task=stsb loss=0.4951
Saved fine-tuned weights to 'multitask_distilbert_finetuned.pt'.

DEMO: 5 test samples